# 01 · Bicycle Dataset & NCDB Online Validation (2010–2023)

This notebook validates the two primary national data sources used in the Canada bicycle collision study:

1. **Public Linked NCDB Microdata (`2010–2021`)**: `155,984` retained input rows (`155,969` distinct normalized rows) across `68,687` unique bicycle-involved casualty collisions (`V_TYPE='17'`), separated into event (`collisions`), vehicle (`vehicles`), and person (`people`) grains in `data/derived/build.duckdb`.
2. **NCDB Online Aggregate Cube (`2010–2023`)**: `4,536` normalized cells across `462` dataset/year/measure partitions (`data/reference/ncdb_online_bicycles_2010_2023.csv`), recording `712` bicyclist deaths and `75,610` nonfatal injuries under `Road User Class=Bicyclist`.
3. **Published 2024 Benchmark (`2020–2023`)**: Official Transport Canada revised bicyclist fatality counts (`53, 42, 51, 46`) compared against the NCDB Online person query (`54, 42, 51, 46`).

In [1]:
import csv
import json
from pathlib import Path
import duckdb
from bicycle_collisions.build import EVENT_KEY, VEHICLE_KEY, PERSON_KEY

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src/bicycle_collisions").exists():
            return candidate
    raise RuntimeError("Could not locate projects/bicycles root")

ROOT = find_project_root()
summary = json.loads((ROOT / "reports/summary.json").read_text())
validation = json.loads((ROOT / "data/derived/validation.json").read_text())

con = duckdb.connect(str(ROOT / "data/derived/build.duckdb"), read_only=True)
n_collisions = con.execute("SELECT count(*) FROM collisions").fetchone()[0]
n_distinct_events = con.execute(f"SELECT count(DISTINCT ({EVENT_KEY})) FROM collisions").fetchone()[0]
n_vehicles = con.execute("SELECT count(*) FROM vehicles").fetchone()[0]
n_distinct_vehicles = con.execute(f"SELECT count(DISTINCT ({VEHICLE_KEY})) FROM vehicles").fetchone()[0]
n_people = con.execute("SELECT count(*) FROM people").fetchone()[0]
n_distinct_people = con.execute(f"SELECT count(DISTINCT ({PERSON_KEY})) FROM people").fetchone()[0]
n_unresolved = con.execute("SELECT count(*) FROM unresolved_records").fetchone()[0]
con.close()

print(f"Project root: {ROOT.name}")
print(f"Input rows: {validation['input_rows']:,} | Distinct normalized rows: {validation['distinct_rows']:,}")
print(f"Unique bicycle collisions (2010-2021): {n_collisions:,} (key unique: {n_collisions == n_distinct_events})")
print(f"Known vehicles in bicycle events:      {n_vehicles:,} (key unique: {n_vehicles == n_distinct_vehicles})")
print(f"Known people in bicycle events:        {n_people:,} (key unique: {n_people == n_distinct_people})")
print(f"Unresolved/dummy identity records:     {n_unresolved:,} (positive casualty rows: {validation['unresolved_positive_casualty_rows']})")
print(f"Person severity conflicts: {validation['person_severity_conflicts']} | Road-user conflicts: {validation['person_road_user_conflicts']}")

Project root: bicycles
Input rows: 155,984 | Distinct normalized rows: 155,969
Unique bicycle collisions (2010-2021): 68,687 (key unique: True)
Known vehicles in bicycle events:      138,532 (key unique: True)
Known people in bicycle events:        155,650 (key unique: True)
Unresolved/dummy identity records:     306 (positive casualty rows: 1)
Person severity conflicts: 3 | Road-user conflicts: 5


## 2. NCDB Online Partition Additivity & Query Context Comparison (2010–2023)

NCDB Online provides two distinct query contexts:
- **Person Context (`Road User Class=Bicyclist`)**: counts casualties among people classified as bicyclists (`712` deaths and `75,610` injured across `2010–2023`), reconciling with age and gender profiles.
- **Vehicle Context (`Vehicle Type=Bicycle`)**: counts bicycle units involved (`76,153` vehicles across `2010–2023`), reconciling with month, weekday, hour, light, weather, surface, and road configuration profiles.

During `2020–2022`, the two query contexts report different casualty totals inside the same August 12, 2025 cube vintage.

In [2]:
from bicycle_collisions.online import validate_partitions

online_csv = ROOT / "data/reference/ncdb_online_bicycles_2010_2023.csv"
with online_csv.open() as f:
    online_rows = [{**r, "year": int(r["year"]), "count": int(r["count"])} for r in csv.DictReader(f)]

partition_totals = validate_partitions(online_rows)
print(f"Normalized online observations: {len(online_rows):,}")
print(f"Verified dataset/year/measure partitions: {len(partition_totals):,}")
print(f"Online totals (2010-2023 Road User=Bicyclist): {summary['online_totals']['fatalities']} deaths, {summary['online_totals']['injured']:,} nonfatal injuries\n")

ctx_rows = list(csv.DictReader((ROOT / "reports/tables/online_query_context_comparison.csv").open()))
divergent = [r for r in ctx_rows if int(r["injury_difference"]) != 0 or int(r["fatality_difference"]) != 0]
print("Years with Person-Query vs. Vehicle-Query Casualty Divergence:")
for r in divergent:
    print(
        f"  {r['year']}: Person injured={int(r['person_query_injured']):,} vs Vehicle injured={int(r['vehicle_query_injured']):,} "
        f"(diff={int(r['injury_difference']):+d}) | Person deaths={r['person_query_fatalities']} vs Vehicle deaths={r['vehicle_query_fatalities']} "
        f"(diff={int(r['fatality_difference']):+d})"
    )

Normalized online observations: 4,536
Verified dataset/year/measure partitions: 462
Online totals (2010-2023 Road User=Bicyclist): 712 deaths, 75,610 nonfatal injuries

Years with Person-Query vs. Vehicle-Query Casualty Divergence:
  2020: Person injured=3,837 vs Vehicle injured=3,440 (diff=+397) | Person deaths=54 vs Vehicle deaths=50 (diff=+4)
  2021: Person injured=4,432 vs Vehicle injured=4,059 (diff=+373) | Person deaths=42 vs Vehicle deaths=39 (diff=+3)
  2022: Person injured=4,281 vs Vehicle injured=3,928 (diff=+353) | Person deaths=51 vs Vehicle deaths=48 (diff=+3)


## 3. Public Record vs. Online Cube & 2024 Published Benchmark Reconciliation

We compare annual bicycle vehicle counts and resolved bicyclist casualties from the `2010–2021` public microdata against NCDB Online (`reports/tables/record_online_comparison.csv`), and compare `2020–2023` bicyclist deaths against Transport Canada's 2024 annual statistics publication (`reports/tables/published_fatality_comparison.csv`).

In [3]:
rec_cmp = list(csv.DictReader((ROOT / "reports/tables/record_online_comparison.csv").open()))
print("Year | Rec Bikes | Online Bikes | Rec Deaths | Online Deaths | Rec Injuries | Online Injuries")
print("-" * 89)
for r in rec_cmp:
    print(
        f"{r['year']} | {int(r['record_bicycle_vehicles']):9,d} | {int(r['online_bicycle_vehicles']):12,d} | "
        f"{int(r['record_resolved_bicyclist_deaths']):10,d} | {int(r['online_bicyclist_deaths']):13,d} | "
        f"{int(r['record_resolved_bicyclist_injuries']):12,d} | {int(r['online_bicyclist_injuries']):15,d}"
    )

pub_cmp = list(csv.DictReader((ROOT / "reports/tables/published_fatality_comparison.csv").open()))
print("\n2020-2023 Published Revised vs. NCDB Online Bicyclist Fatalities:")
for r in pub_cmp:
    diff = int(r["online_fatalities"]) - int(r["published_fatalities"])
    print(f"  {r['year']}: Published={r['published_fatalities']} vs Online={r['online_fatalities']} (diff={diff:+d})")

Year | Rec Bikes | Online Bikes | Rec Deaths | Online Deaths | Rec Injuries | Online Injuries
-----------------------------------------------------------------------------------------
2010 |     6,711 |        6,712 |         61 |            61 |        6,483 |           6,484
2011 |     6,586 |        6,587 |         56 |            56 |        6,415 |           6,416
2012 |     6,856 |        6,842 |         62 |            63 |        6,617 |           6,614
2013 |     6,450 |        6,435 |         69 |            69 |        6,253 |           6,251
2014 |     5,804 |        5,799 |         42 |            42 |        5,574 |           5,581
2015 |     6,461 |        6,455 |         49 |            50 |        6,208 |           6,200
2016 |     6,319 |        6,341 |         49 |            49 |        6,080 |           6,102
2017 |     5,567 |        5,600 |         40 |            38 |        5,381 |           5,428
2018 |     4,877 |        4,892 |         45 |            48 |  